# Episode 16: GRPO: reinforcement learning without a critic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep016-grpo.ipynb)

Our model writes eight stories per instruction, and each is scored against the others, in place of PPO's critic, with group relative policy optimisation, the reinforcement learning behind DeepSeek-R1.

[The video](https://youtu.be/SoFl6r8Qz-0) · [The episode's page](https://mvgpt.nenuadrian.com/ep016-grpo/) · [The docs](https://nenuadrian.github.io/mvGPT/grpo/) · [The code at `ep016/end`](https://github.com/nenuadrian/mvGPT/tree/ep016/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep016/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("sft", "ppo/kl-0.05")

## The data

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids

## GRPO
Eight stories per instruction and no critic: ten minutes with a KL weight of 0.05, then ten
with none.

In [ ]:
!python grpo.py 10 0.05
!python grpo.py 10 0

## Against SFT and PPO, and stories

In [ ]:
!python evaluate.py checkpoints/sft/model.pt checkpoints/ppo/kl-0.05/model.pt checkpoints/grpo/kl-0.05/model.pt checkpoints/grpo/kl-0.0/model.pt --out results/grpo
!python sample.py checkpoints/grpo/kl-0.05/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3
!python sample.py checkpoints/grpo/kl-0.0/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3

Next: [episode 17, RMSNorm, SwiGLU, QK-norm: a modern GPT block](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep017-modern-block.ipynb)